# C3 - Day 7: Model 2 evaluation - does it stay inside the C3 boundary?
**AnchorAI | Component 3: Cognitive Checkpointing**

Three systems write a restart summary for the same **72 unseen test checkpoints** (task families the model never saw in training):

| System | What it is |
|--------|------------|
| **Template** | The rule-based summary used in the prototype (restates the fields) |
| **Base Qwen** | Qwen2.5-1.5B-Instruct with the same instructions, *not* fine-tuned |
| **Fine-tuned** | Qwen2.5-1.5B-Instruct + our LoRA adapter |

**What we measure.** ROUGE-L against a synthetic reference is only a sanity check, because the references come from the same generator as the training data. The real questions are about the C3 boundary:

* **Own words** - no words that are not in the checkpoint or in the fixed summary wording (no new facts, no chatty extras).
* **No advice** - no advice, suggestions, code or answers.
* **Right depth** - Low / Medium / High use the right fields (and leave out the others).
* **Second person** - "I need to" is rewritten as "you need to".
* **Question kept** - a student's question is repeated as an open question and not answered.
* **Length** - close to the reference length for that checkpoint.

Each system is allowed its own fixed wording (for the template: its labels such as "Already done"); everything else must come from the student.

Finally, a **blinded human rating** of 18 pairs (base vs fine-tuned) is prepared for you to fill in.

> Runtime: **T4 GPU**. Needs `qwen_c3_lora` in `MyDrive/C3/` (from notebook 06) and the data files in `MyDrive/C3/data/generated/`.

## Step 0 - Setup and data

In [ ]:
import os
DRY_RUN = os.environ.get('C3_DRY_RUN') == '1'   # used only by the local tests, never set in Colab
if not DRY_RUN:
    import torch
    assert torch.cuda.is_available(), 'No GPU found. Use Runtime -> Change runtime type -> T4 GPU.'
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('DRY RUN (local test, no model)')

In [ ]:
if not DRY_RUN:
    !pip -q install peft bitsandbytes accelerate

In [ ]:
import os, json, random, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/C3'
    print('Running in Colab - Drive mounted.')
except ImportError:
    BASE = os.environ.get('C3_BASE', r'C:\Users\user\OneDrive\Documents\AnchorAI\AnchorAI\components\c3_checkpointing')
    print('Running locally.')

GEN_DIR = os.path.join(BASE, 'data', 'generated')
OUTPUT_DIR = os.path.join(BASE, 'outputs')
ADAPTER_DIR = os.path.join(BASE, 'qwen_c3_lora')
os.makedirs(OUTPUT_DIR, exist_ok=True)
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

def read_jsonl(name):
    with open(os.path.join(GEN_DIR, name), encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

test_rows = read_jsonl('restart_summary_test.jsonl')
frame = json.load(open(os.path.join(GEN_DIR, 'frame_vocab.json'), encoding='utf-8'))
FRAME_VOCAB = set(frame['frame_vocab'])
SYSTEM_PROMPT = frame['system_prompt']
print(len(test_rows), 'test examples from families:', sorted({r['family'] for r in test_rows}))

## Step 1 - Helper code (the same modules the local tests check)
Text helpers, the metrics, the difficulty rule, the template summary and the generation function.

In [ ]:
"""Small text helpers shared by the data generator, the metrics and the Colab notebooks (no dependencies)."""

import re

_PRONOUNS = [
    (r"\bi'm\b", "you're"), (r"\bi've\b", "you've"), (r"\bi'll\b", "you'll"), (r"\bi'd\b", "you'd"),
    (r"\bi am\b", "you are"), (r"\bi was\b", "you were"), (r"\bi\b", "you"), (r"\bmyself\b", "yourself"),
    (r"\bmy\b", "your"), (r"\bmine\b", "yours"), (r"\bme\b", "you"),
]


def to_you(text: str) -> str:
    out = text
    for pattern, repl in _PRONOUNS:
        out = re.sub(pattern, repl, out, flags=re.IGNORECASE)
    return out


def cap(text: str) -> str:
    text = text.strip()
    if not text:
        return text
    first = text.split()[0]
    if any(ch in first for ch in "._0123456789") or first != first.lower():
        return text
    return text[0].upper() + text[1:]


def sentence(text: str) -> str:
    text = text.strip().rstrip(".,;:!")
    return text if text.endswith("?") else text + "."


def tokens(text: str) -> list[str]:
    return [t for t in (x.strip(".'-") for x in re.findall(r"[a-z0-9_.'\-]+", text.lower())) if t]


"""Evaluation metrics for restart summaries.

The central question is not "does it match a reference" but "does it stay inside the C3 boundary":
  * grounding  - every content word comes from the checkpoint (or from the fixed summary wording),
  * advice     - no advice, suggestions, code or answers,
  * depth      - Low / Medium / High use the right fields,
  * voice      - first person ("I need to") is rewritten as second person ("you need to"),
  * question   - a student's question is repeated, not answered,
  * length     - close to the reference length for the same checkpoint.
ROUGE-L against the synthetic reference is reported only as a sanity check.
"""

import re


ADVICE_PATTERNS = [
    r"\byou (should|could|might want to|may want to|can try|can use|need to consider)\b",
    r"\b(i recommend|i suggest|i would|consider|try to|try using|make sure|best practice|a good idea)\b",
    r"\b(here is|here's|for example|e\.g\.|for instance)\b",
    r"\bthe (answer|solution) is\b",
    r"```",
    r"\b(def |class |select .* from |import |console\.log|function\()",
]
_ADVICE = [re.compile(p, re.IGNORECASE) for p in ADVICE_PATTERNS]


def _norm(text: str) -> str:
    return " ".join(tokens(text))


def grounding(output: str, input_text: str, frame_vocab: set[str]) -> dict:
    known = set(tokens(input_text)) | frame_vocab
    toks = tokens(output)
    ungrounded = [t for t in toks if t not in known]
    return {"n_tokens": len(toks), "ungrounded": ungrounded, "rate": (len(ungrounded) / len(toks)) if toks else 0.0}


def advice_hits(output: str, input_text: str = "") -> list[str]:
    """Patterns found in the output but not in the student's own text (what the student wrote is never 'advice')."""
    return [p.pattern for p in _ADVICE if p.search(output) and not p.search(input_text)]


def contains_text(output: str, field_text: str | None) -> bool:
    """True if the field appears in the output, as written or with first person rewritten as second person."""
    if not field_text:
        return False
    out = _norm(output)
    return _norm(to_you(field_text)) in out or _norm(field_text) in out


FIRST_PERSON = {"i", "my", "me", "mine", "myself", "i'm", "i've", "i'll", "i'd"}


def first_person_words(output: str) -> list[str]:
    """First-person words left in the output (the summary should be written to the student as 'you')."""
    return [t for t in tokens(output) if t in FIRST_PERSON]


def depth_policy(output: str, fields: dict, level: str) -> dict:
    """Low: only working point + next step. Medium: adds goal and problem. High: adds completed, resources, open items."""
    allowed = {"Low": {"working_point", "next_step"},
               "Medium": {"working_point", "next_step", "goal", "problem"},
               "High": {"working_point", "next_step", "goal", "problem", "completed", "resources", "open_items"}}[level]
    expected = {k for k in allowed if fields.get(k)}
    present, leaked = set(), set()
    for key in ("goal", "completed", "working_point", "problem", "next_step", "resources"):
        if contains_text(output, fields.get(key)):
            present.add(key)
    items = fields.get("open_items") or []
    # A short item such as "conclusion" can occur inside another sentence, so the list is recognised by its heading.
    heading = "still open in your files" in _norm(output) or "open items found in your own files" in _norm(output)
    if heading and (not items or all(contains_text(output, i) for i in items)):
        present.add("open_items")
    for key in ("goal", "completed", "working_point", "problem", "next_step", "resources", "open_items"):
        if key in present and key not in allowed:
            leaked.add(key)
    missing = expected - present
    return {"missing": sorted(missing), "leaked": sorted(leaked), "ok": not missing and not leaked}


def question_handling(output: str, fields: dict, level: str, input_text: str = "") -> dict | None:
    """Only defined when a question was written and the level should mention it."""
    q = fields.get("problem")
    if not q or not q.strip().endswith("?") or level == "Low":
        return None
    restated = contains_text(output, q)
    answered = bool(advice_hits(output, input_text))
    return {"restated": restated, "answered": answered, "ok": restated and not answered}


def length_ok(output: str, reference: str, lo: float = 0.5, hi: float = 1.6) -> bool:
    n, r = len(output.split()), len(reference.split())
    return r > 0 and lo * r <= n <= hi * r


def rouge_l(candidate: str, reference: str) -> float:
    a, b = tokens(candidate), tokens(reference)
    if not a or not b:
        return 0.0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b, start=1):
            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))
        prev = cur
    lcs = prev[-1]
    if lcs == 0:
        return 0.0
    p, r = lcs / len(a), lcs / len(b)
    return 2 * p * r / (p + r)


def score_example(output: str, row: dict, frame_vocab: set[str]) -> dict:
    g = grounding(output, row["input"], frame_vocab)
    pol = depth_policy(output, row["fields"], row["level"])
    q = question_handling(output, row["fields"], row["level"], row["input"])
    hits = advice_hits(output, row["input"])
    return {
        "ungrounded_rate": g["rate"],
        "grounded_ok": g["rate"] <= 0.02,
        "advice": hits,
        "no_advice": not hits,
        "policy_ok": pol["ok"],
        "second_person_ok": not first_person_words(output),
        "question_ok": None if q is None else q["ok"],
        "length_ok": length_ok(output, row["output"]),
        "rouge_l": rouge_l(output, row["output"]),
    }


"""Transparent resumption-difficulty rule. Mirrors app/static/core.js (checked by shared test vectors)."""

import math
from typing import Any

LEVELS = ("Low", "Medium", "High")
COMPLEXITY = {"simple": 0, "moderate": 1, "complex": 2}
MAX_SCORE = 8

DEPTH = {
    "Low": {"name": "Short", "suggested": ["workingPoint", "nextStep"]},
    "Medium": {"name": "Standard", "suggested": ["goal", "workingPoint", "problem", "nextStep"]},
    "High": {"name": "Full", "suggested": ["goal", "completed", "workingPoint", "problem", "nextStep", "resources"]},
}


def _valid_minutes(minutes: Any) -> float | None:
    try:
        value = float(minutes)
    except (TypeError, ValueError):
        return None
    if math.isnan(value) or value < 0:
        return None
    return value


def format_minutes(minutes: Any) -> str:
    m = _valid_minutes(minutes)
    if m is None:
        return "an unknown time"
    if m < 1:
        return "less than a minute"
    if m < 60:
        return f"{round(m)} min"
    if m < 1440:
        h = m / 60
        return f"{int(h) if float(h).is_integer() else round(h, 1)} h"
    d = m / 1440
    if float(d).is_integer():
        return "1 day" if int(d) == 1 else f"{int(d)} days"
    return f"{round(d, 1)} days"


def duration_points(minutes: Any) -> int:
    m = _valid_minutes(minutes)
    if m is None or m < 15:
        return 0
    if m < 60:
        return 1
    if m < 240:
        return 2
    return 3


def open_item_points(n: int) -> int:
    if n >= 5:
        return 2
    if n >= 2:
        return 1
    return 0


def predict_difficulty(minutes: Any, complexity: str = "moderate", open_items: Any = 0, type: str = "other") -> dict:
    """Hand-set rule. Direction (longer interruption, more complex task -> harder) follows Model 1b;
    thresholds are placeholders until pilot data recalibrates them."""
    complexity = complexity if complexity in COMPLEXITY else "moderate"
    try:
        open_items = max(0, int(float(open_items or 0)))
    except (TypeError, ValueError):
        open_items = 0
    switches = type == "other_task"

    d = duration_points(minutes)
    c = COMPLEXITY[complexity]
    o = open_item_points(open_items)
    s = 1 if switches else 0
    score = d + c + o + s
    level = "High" if score >= 5 else "Medium" if score >= 3 else "Low"

    m = _valid_minutes(minutes)
    reasons = [
        {"text": f"Away about {format_minutes(m)}" if m is not None else "Time away not given", "points": d},
        {"text": f"Task complexity: {complexity}", "points": c},
        {"text": f"{open_items} unfinished item{'' if open_items == 1 else 's'}", "points": o},
        {
            "text": "Interrupted by another work task" if switches else "Interruption does not need another work task",
            "points": s,
        },
    ]
    return {"score": score, "max": MAX_SCORE, "level": level, "depth": DEPTH[level], "reasons": reasons}


"""Template restart summary. It only restates the student's own checkpoint text (never adds content).

This is the baseline generator: the learned Qwen model (Model 2) is compared against it.
Mirrors app/static/core.js (checked by shared test vectors).
"""

from typing import Any


FIELD_LABELS = {
    "goal": "Goal",
    "completed": "Completed work",
    "workingPoint": "Current working point",
    "problem": "Current problem or decision",
    "nextStep": "Next step",
    "resources": "Files / resources",
}


def _clean(value: Any) -> str:
    return str(value or "").strip()


def build_summary(task_title: str, checkpoint: dict, level: str, away_minutes: float) -> dict:
    cp = checkpoint or {}
    lines: list[dict] = []

    def add(label: str, text: Any) -> None:
        t = _clean(text)
        if t:
            lines.append({"label": label, "text": t})

    add("You stopped at", cp.get("workingPoint"))
    add("Next step", cp.get("nextStep"))
    if level in ("Medium", "High"):
        add("Goal", cp.get("goal"))
        add("Open problem or decision", cp.get("problem"))
    if level == "High":
        add("Already done", cp.get("completed"))
        add("Files / resources", cp.get("resources"))
        items = [str(i) for i in (cp.get("openItems") or [])][:5]
        if items:
            add("Open items found in your own files", "; ".join(items))

    title = task_title or "your task"
    intro = f'Welcome back to "{title}". You were away about {format_minutes(away_minutes)}.'
    return {"intro": intro, "lines": lines, "level": level, "depth": DEPTH[level]}


def summary_to_text(summary: dict) -> str:
    body = "\n".join(f"{line['label']}: {line['text']}" for line in summary["lines"])
    return summary["intro"] + ("\n\n" + body if body else "\n\n(No checkpoint text was written.)")


In [ ]:
"""LoRA training and generation helpers for Model 2.

This file is used twice: the Colab notebooks include its source, and the local tests run the same code
on a tiny random model, so a mistake here is caught before any GPU time is spent.
"""
import inspect
import time

import torch
from torch.utils.data import Dataset
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments, set_seed

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def encode_row(tok, row, max_len=1024):
    """Tokenise one chat example; only the assistant answer is trained on (the prompt is masked with -100)."""
    prompt = tok.apply_chat_template(row["messages"][:2], tokenize=False, add_generation_prompt=True)
    full = tok.apply_chat_template(row["messages"], tokenize=False)
    if not full.startswith(prompt):
        raise ValueError("chat template does not start with the prompt for " + row.get("id", "?"))
    prompt_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tok(full, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    n = min(len(full_ids), max_len)
    return {"input_ids": full_ids[:n], "attention_mask": [1] * n, "labels": labels[:n]}


class ListDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def attach_lora(model, quantized, r=16, alpha=32, dropout=0.05):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    if quantized:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    config = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, bias="none", task_type="CAUSAL_LM",
                        target_modules=LORA_TARGETS)
    return get_peft_model(model, config)


def training_arguments(output_dir, epochs, lr, batch_size, accum, has_eval, fp16, seed):
    kwargs = dict(
        output_dir=output_dir, num_train_epochs=epochs, per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size, gradient_accumulation_steps=accum, learning_rate=lr,
        lr_scheduler_type="cosine", warmup_steps=5, logging_steps=5, save_strategy="no", report_to="none",
        fp16=fp16, seed=seed, remove_unused_columns=False,
    )
    if has_eval:
        accepted = inspect.signature(TrainingArguments.__init__).parameters
        kwargs["eval_strategy" if "eval_strategy" in accepted else "evaluation_strategy"] = "epoch"
    return TrainingArguments(**kwargs)


def train(model, tok, train_rows, val_rows, out_dir, epochs=5, lr=2e-4, batch_size=4, accum=4, fp16=True,
          max_len=1024, seed=42):
    set_seed(seed)
    train_items = [encode_row(tok, r, max_len) for r in train_rows]
    val_items = [encode_row(tok, r, max_len) for r in val_rows]
    collator = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100)
    args = training_arguments(out_dir, epochs, lr, batch_size, accum, bool(val_items), fp16, seed)
    trainer = Trainer(model=model, args=args, train_dataset=ListDataset(train_items),
                      eval_dataset=ListDataset(val_items) if val_items else None, data_collator=collator)
    model.config.use_cache = False
    start = time.time()
    result = trainer.train()
    model.config.use_cache = True
    info = {
        "seconds": round(time.time() - start, 1), "train_loss": float(result.training_loss),
        "epochs": epochs, "learning_rate": lr, "batch_size": batch_size, "grad_accumulation": accum,
        "max_len": max_len, "seed": seed, "n_train": len(train_items), "n_val": len(val_items),
        "longest_example_tokens": max(len(i["input_ids"]) for i in train_items),
        "history": trainer.state.log_history,
    }
    return trainer, info


@torch.no_grad()
def generate_batch(model, tok, message_lists, max_new_tokens=300, batch_size=8):
    """Greedy generation for a list of chat prompts (system + user messages); returns the answers."""
    model.eval()
    tok.padding_side = "left"
    answers = []
    for i in range(0, len(message_lists), batch_size):
        chunk = message_lists[i:i + batch_size]
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        for j in range(len(chunk)):
            new_tokens = out[j][enc["input_ids"].shape[1]:]
            answers.append(tok.decode(new_tokens, skip_special_tokens=True).strip())
    return answers


## Step 2 - Load the base model and the adapter
The same model object is used for both neural systems: with the adapter switched on it is the *fine-tuned* model, inside `disable_adapter()` it is the *base* model.

In [ ]:
if not DRY_RUN:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.float16)
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map='auto')
    model = PeftModel.from_pretrained(base, ADAPTER_DIR)
    print('Loaded base model + adapter from', ADAPTER_DIR)

## Step 3 - Write the summaries
Greedy decoding (no randomness), so the result is reproducible. The base model receives exactly the same instructions as the fine-tuned one.

In [ ]:
def to_camel(fields):
    return {'goal': fields.get('goal'), 'completed': fields.get('completed'), 'workingPoint': fields.get('working_point'),
            'problem': fields.get('problem'), 'nextStep': fields.get('next_step'), 'resources': fields.get('resources'),
            'openItems': fields.get('open_items') or []}

def template_output(row):
    title = row['input'].split('\n')[0].replace('Task: ', '')
    return summary_to_text(build_summary(title, to_camel(row['fields']), row['level'], row['minutes']))

prompts = [r['messages'][:2] for r in test_rows]
outputs = {'Template': [template_output(r) for r in test_rows]}

if DRY_RUN:
    # stand-ins so the evaluation code can be tested without a GPU
    rng = random.Random(0)
    outputs['Fine-tuned'] = [r['output'] for r in test_rows]
    outputs['Base Qwen'] = [r['output'] + ' You should try a different approach, for example add more tests.' if rng.random() < 0.5
                            else 'Sure! Here is a summary: ' + r['output'] for r in test_rows]
else:
    outputs['Fine-tuned'] = generate_batch(model, tok, prompts, max_new_tokens=300, batch_size=8)
    with model.disable_adapter():
        outputs['Base Qwen'] = generate_batch(model, tok, prompts, max_new_tokens=300, batch_size=8)
print({k: len(v) for k, v in outputs.items()})

## Step 4 - Score every summary

In [ ]:
# the template has its own fixed labels; they are allowed for the template only
TEMPLATE_VOCAB = set()
for r in read_jsonl('restart_summary_train.jsonl'):
    known = set(tokens(r['input']))
    TEMPLATE_VOCAB |= {t for t in tokens(template_output(r)) if t not in known}

rows = []
for system, outs in outputs.items():
    vocab = FRAME_VOCAB | TEMPLATE_VOCAB if system == 'Template' else FRAME_VOCAB
    for r, out in zip(test_rows, outs):
        s = score_example(out, r, vocab)
        rows.append({'system': system, 'id': r['id'], 'level': r['level'], **{k: v for k, v in s.items() if k != 'advice'}})
scores = pd.DataFrame(rows)

def pct(x):
    x = x.dropna()
    return round(100 * x.astype(float).mean(), 1) if len(x) else float('nan')

summary = scores.groupby('system').agg(
    own_words=('grounded_ok', pct), no_advice=('no_advice', pct), depth_ok=('policy_ok', pct),
    second_person=('second_person_ok', pct), question_ok=('question_ok', pct), length_ok=('length_ok', pct),
    ungrounded_word_rate=('ungrounded_rate', lambda x: round(100 * x.mean(), 1)),
    rouge_l=('rouge_l', lambda x: round(x.mean(), 3))).loc[['Template', 'Base Qwen', 'Fine-tuned']]
print('Share of summaries that pass each check (%), 72 unseen test checkpoints:')
print(summary.to_string())
print('\nBy difficulty level (depth_ok %):')
print(scores.pivot_table(index='system', columns='level', values='policy_ok', aggfunc=lambda x: pct(x))[['Low', 'Medium', 'High']].to_string())

In [ ]:
checks = ['own_words', 'no_advice', 'depth_ok', 'second_person', 'question_ok', 'length_ok']
fig, ax = plt.subplots(figsize=(10, 4))
width = 0.26
for i, system in enumerate(['Template', 'Base Qwen', 'Fine-tuned']):
    ax.bar(np.arange(len(checks)) + (i - 1) * width, [summary.loc[system, c] for c in checks], width, label=system)
ax.set_xticks(np.arange(len(checks))); ax.set_xticklabels(['Own words', 'No advice', 'Right depth', 'Second person', 'Question kept', 'Length ok'])
ax.set_ylabel('% of test summaries passing'); ax.set_ylim(0, 105); ax.set_title('Model 2: staying inside the C3 boundary'); ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model2_evaluation.png'), dpi=150)
plt.show()

## Step 5 - Look at real outputs (same checkpoint, three systems)
Read these yourself: this is what you will show in the presentation.

In [ ]:
for lvl in ('Low', 'Medium', 'High'):
    i = next(k for k, r in enumerate(test_rows) if r['level'] == lvl and (r['fields'].get('problem') or '').endswith('?'))
    print('=' * 90); print(lvl, '|', test_rows[i]['family']); print(test_rows[i]['input'])
    for system in ('Template', 'Base Qwen', 'Fine-tuned'):
        print(f'\n--- {system} ---'); print(outputs[system][i])

## Step 6 - Human rating sheet (blinded)
18 test checkpoints (6 per level). For each, two summaries (A and B) appear in random order; you do not know which one is fine-tuned. Open `human_rating_sheet.csv` (in `MyDrive/C3/outputs/`), fill the columns, save, and run Step 7.

* `better`: A, B or same
* `*_faithful`: 1-5, does it say only what the student wrote?
* `*_clear`: 1-5, is it easy to resume from?

In [ ]:
rng = random.Random(7)
picked = []
for lvl in ('Low', 'Medium', 'High'):
    idx = [k for k, r in enumerate(test_rows) if r['level'] == lvl]
    picked += rng.sample(idx, 6)
rng.shuffle(picked)
sheet, key = [], {}
for k in picked:
    a_is_ft = rng.random() < 0.5
    A = outputs['Fine-tuned'][k] if a_is_ft else outputs['Base Qwen'][k]
    B = outputs['Base Qwen'][k] if a_is_ft else outputs['Fine-tuned'][k]
    sheet.append({'id': test_rows[k]['id'], 'level': test_rows[k]['level'], 'checkpoint': test_rows[k]['input'],
                  'summary_A': A, 'summary_B': B, 'better': '', 'A_faithful': '', 'B_faithful': '', 'A_clear': '', 'B_clear': '', 'comment': ''})
    key[test_rows[k]['id']] = {'A': 'Fine-tuned' if a_is_ft else 'Base Qwen', 'B': 'Base Qwen' if a_is_ft else 'Fine-tuned'}
pd.DataFrame(sheet).to_csv(os.path.join(OUTPUT_DIR, 'human_rating_sheet.csv'), index=False, encoding='utf-8-sig')
json.dump(key, open(os.path.join(OUTPUT_DIR, 'human_rating_key.json'), 'w'), indent=1)
print('Wrote human_rating_sheet.csv and human_rating_key.json to', OUTPUT_DIR)

## Step 7 - Read your ratings (run this after you filled the sheet)

In [ ]:
path = os.path.join(OUTPUT_DIR, 'human_rating_sheet.csv')
filled = pd.read_csv(path, encoding='utf-8-sig')
filled = filled[filled['better'].astype(str).str.upper().isin(['A', 'B', 'SAME'])]
key = json.load(open(os.path.join(OUTPUT_DIR, 'human_rating_key.json')))
if filled.empty:
    print('The sheet is not filled yet. Fill the "better" column (A / B / same) and the 1-5 scores, then run this cell again.')
else:
    wins = {'Fine-tuned': 0, 'Base Qwen': 0, 'same': 0}
    faithful = {'Fine-tuned': [], 'Base Qwen': []}
    clear = {'Fine-tuned': [], 'Base Qwen': []}
    for _, r in filled.iterrows():
        k = key[r['id']]
        b = str(r['better']).upper()
        wins['same' if b == 'SAME' else k[b]] += 1
        for side in ('A', 'B'):
            faithful[k[side]].append(float(r[side + '_faithful'])); clear[k[side]].append(float(r[side + '_clear']))
    print(f'{len(filled)} pairs rated. Preferred:', wins)
    for s in ('Fine-tuned', 'Base Qwen'):
        print(f'{s}: faithful {np.mean(faithful[s]):.2f}/5, clear {np.mean(clear[s]):.2f}/5')

## Step 8 - Save the results

In [ ]:
summary.to_csv(os.path.join(OUTPUT_DIR, 'results_model2_evaluation.csv'))
scores.to_csv(os.path.join(OUTPUT_DIR, 'results_model2_per_example.csv'), index=False)
with open(os.path.join(OUTPUT_DIR, 'model2_test_outputs.json'), 'w', encoding='utf-8') as f:
    json.dump({'ids': [r['id'] for r in test_rows], 'outputs': outputs}, f, ensure_ascii=False, indent=1)
print('Saved results_model2_evaluation.csv, results_model2_per_example.csv, model2_test_outputs.json')

## Summary - Day 7
Read the table in Step 4:
* **Own words / No advice / Question kept** are the evidence for the C3-C4 boundary: the summary restates the student's own words and does not answer or advise.
* **Right depth** shows that the Low / Medium / High policy is learned, not just prompted.
* Compare the base model: if it adds advice or ignores the length rule, fine-tuning is what makes the behaviour reliable.
* Be honest in the viva: the test data is synthetic and the references come from the same generator, so ROUGE-L is a sanity check; the human ratings and the boundary checks are the main evidence.

**Next: connect the model to the backend (`SummaryGenerator`) and record the demo.**